In [2]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

# Ordered Binary Trees

This notebook implements *ordered binary trees*.  In order to define this notion, we first have to define 
the concept of *ordered binary trees*.  In the following, assume a set $\texttt{Key}$ and a set $\texttt{Value}$ are given.   Then, the
set $\mathcal{B}$ of all *ordered binary trees* is defined inductively.
  - $\texttt{Nil} \in \mathcal{B}$
  - $\texttt{Node}(k, v, l, r) \in \mathcal{B}$  iff the following conditions hold:
      - $k \in\texttt{Key}$,
      - $v \in\texttt{Value}$,
      - $l \in\mathcal{B}$,
      - $r \in\mathcal{B}$,
      - all keys that occur in the left subtree $l$ are smaller than $k$,
      - all keys that occur in the right subtree $r$ are bigger than $k$,
      
        therefore $l < k < r$.

The class `OrderedBinaryTree` represents the nodes of an ordered binary tree.
  - $\texttt{Nil}$           is created as $\texttt{OrderedBinaryTree}()$.
  - $\texttt{Node}(k,v,l,r)$ is created as follows:
    ```
    t = OrderedBinaryTree()
    t.mKey   = k
    t.mValue = v
    t.mLeft  = l
    t.mRight = r
    ```
The constructor below creates the empty tree.

In [3]:
class OrderedBinaryTree<Key, Value> {
    mKey: Key | null;
    mValue: Value | null;
    mLeft: OrderedBinaryTree<Key, Value> | null;
    mRight: OrderedBinaryTree<Key, Value> | null;
    mID: number;
    static sNodeCount: number = 0;
    constructor() {
        this.mKey = null;
        this.mValue = null;
        this.mLeft = null;
        this.mRight = null;
    }
}
interface OrderedBinaryTree<Key, Value> {
    mKey: Key | null;
    mValue: Value | null;
    mLeft: OrderedBinaryTree<Key, Value> | null;
    mRight: OrderedBinaryTree<Key, Value> | null;
    isEmpty(): boolean;
    insert(key: Key, value: Value): void;
    find(key: Key): Value | undefined;
    delete(key: Key): void;
    toString(): string;
    toDot(): string;
    _delMin(): [OrderedBinaryTree<Key, Value>, Key, Value];
    _update(tree: OrderedBinaryTree<Key, Value>): void;
    _keyList(): Key[];
    _assignIDs(nodeDict: Record<number, OrderedBinaryTree<Key, Value>>): void;
}

Given an ordered binary tree $t$, the expression $t.\texttt{isEmpty}()$ checks whether $t$ is the empty tree.

In [4]:
OrderedBinaryTree.prototype.isEmpty = function<Key, Value>(this: OrderedBinaryTree<Key, Value>): boolean {
    return this.mKey === null;
};

[Function (anonymous)]


Given an ordered binary tree $t$ and a key $k$, the expression $t.\texttt{find}(k)$ returns the value stored unter the key $k$.
The method `find` is defined inductively as follows:
  - $\texttt{Nil}.\texttt{find}(k) = \Omega$,

    because the empty tree is interpreted as the empty map.
  - $\texttt{Node}(k, v, l, r).\texttt{find}(k) = v$,
      
    because the node $\texttt{Node}(k,v,l,r)$ stores the assignment $k \mapsto v$.
  - $k_1 < k_2 \rightarrow \texttt{Node}(k_2, v, l, r).\texttt{find}(k_1) = l.\texttt{find}(k_1)$,

    because if $k_1$ is less than $k_2$, then any mapping for $k_1$ has to be stored in the left subtree  $l$.
  - $k_1 > k_2 \rightarrow \texttt{Node}(k_2, v, l, r).\texttt{find}(k_1) = r.\texttt{find}(k_1)$,

    because if $k_1$ is greater than $k_2$, then any mapping for $k_1$ has to be stored in the right subtree  $r$.

In [5]:
OrderedBinaryTree.prototype.find = function<Key, Value>(this: OrderedBinaryTree<Key, Value>, key: Key): Value | undefined {
    if (this.isEmpty()) {
        return undefined;
    } else if (this.mKey === key) {
        return this.mValue;
    } else if (key < this.mKey) {
        return this.mLeft.find(key);
    } else {
        return this.mRight.find(key);
    }
};

[Function (anonymous)]


Given an ordered binary tree $t$, a key $k$ and a value $v$, the expression $t.\texttt{insert}(k, v)$ updates the tree $t$ such that the key $k$ is associated with the value $v$.
The method `insert` is defined inductively as follows:
  - $\texttt{Nil}.\texttt{insert}(k,v) = \texttt{Node}(k,v, \texttt{Nil}, \texttt{Nil})$,
  
    If the tree is empty, the information to be stored is stored at the root.
  - $\texttt{Node}(k,v_2,l,r).\texttt{insert}(k,v_1) = \texttt{Node}(k, v_1, l, r)$,

    If the key $k$ is located at the root, we overwrite the old information. 
  - $k_1 < k_2 \rightarrow 
    \texttt{Node}(k_2, v_2, l, r).\texttt{insert}(k_1, v_1) = \texttt{Node}\bigl(k_2, v_2, l.\texttt{insert}(k_1, v_1), r\bigr)$,

    If the key $k_1$, which is the key for which we want to store a value, is less than the key
    $k_2$ at the root, then we have to insert the information in the left subtree.
  - $k_1 > k_2 \rightarrow 
         \texttt{Node}(k_2, v_2, l, r).\texttt{insert}(k_1, v_1) = 
         \texttt{Node}\bigl(k_2, v_2, l, r.\texttt{insert}(k_1, v_1)\bigr)$,

    If the key $k_1$, which is the key for which we want to store a value, is bigger than the key
    $k_2$ at the root, then we have to insert the information in the right subtree.

In [6]:
OrderedBinaryTree.prototype.insert = function<Key, Value>(this: OrderedBinaryTree<Key, Value>, key: Key, value: Value): void {
    if (this.isEmpty()) {
        this.mKey = key;
        this.mValue = value;
        this.mLeft = new OrderedBinaryTree<Key, Value>();
        this.mRight = new OrderedBinaryTree<Key, Value>();
    } else if (this.mKey === key) {
        this.mValue = value;
    } else if (key < this.mKey) {
        this.mLeft.insert(key, value);
    } else {
        this.mRight.insert(key, value);
    }
};

[Function (anonymous)]


Given an ordered binary tree $t$ and a key $k$, the expression $t.\texttt{delete}(k)$ removes the key $k$ and its associated value from $t$.  The method `delete` is defined inductively.
  - $\texttt{Nil}.\texttt{delete}(k) = \texttt{Nil}$.
  - $\texttt{Node}(k,v,\texttt{Nil},r).\texttt{delete}\bigl(k\bigr) = r$.
  - $\texttt{Node}(k,v,l,\texttt{Nil}).\texttt{delete}(k) = l$.
  - If $l \not= \texttt{Nil} \,\wedge\, r \not= \texttt{Nil} \,\wedge\, r.\texttt{delMin}() = [r',k_{min}, v_{min}]$,
    then
    
    $$\texttt{Node}(k,v,l,r).\texttt{delete}(k) = \texttt{Node}(k_{min},v_{min},l,r').$$
      
    If the key to be removed is found at the root of the tree and neither of its subtrees is
    empty, the call  $r\mathtt{.}\texttt{delMin}()$ removes the smallest key together with its
    associated value from the subtree $r$ yielding the subtree $r'$.
    The smallest key from $r$ is then stored at the root of the new tree.

  - $k_1 < k_2 \rightarrow \texttt{Node}(k_2,v_2,l,r).\texttt{delete}\bigl(k_1) = 
    \texttt{Node}(k_2,v_2,l.\texttt{delete}(k_1),r)$.

    If the key that is to be removed is less than the key stored at the root, the key $k$ can only be
    located in the left subtree $l$.  Hence, $k$ is removed from the left subtree $l$ recursively.
  - $k_1 > k_2 \rightarrow \texttt{Node}(k_2,v_2,l,r).\texttt{delete}(k_1) = 
    \texttt{Node}(k_2,v_2,l,r.\texttt{delete}(k_1))$.

    If the key that is to be removed is greater than the key stored at the root, the key $k$ can only be
    located in the right subtree $r$.  Hence, $k$ is removed from the right subtree $r$ recursively.

In [7]:
OrderedBinaryTree.prototype.delete = function<Key, Value>(this: OrderedBinaryTree<Key, Value>, key: Key): void {
    if (this.isEmpty()) {
        return;
    }
    if (key === this.mKey) {
        if (this.mLeft.isEmpty()) {
            this._update(this.mRight);
        } else if (this.mRight.isEmpty()) {
            this._update(this.mLeft);
        } else {
            const [rs, km, vm] = this.mRight._delMin();
            this.mKey = km;
            this.mValue = vm;
            this.mRight = rs;
        }
    } else if (key < this.mKey) {
        this.mLeft.delete(key);
    } else {
        this.mRight.delete(key);
    }
};

[Function (anonymous)]


Given a non-empty ordered binary tree $t$, the expression $t.\texttt{delMin}()$ removes the smallest key $k_m$ and its associated value $v_m$ from $t$ and returns the triple
$$(r,k_m,v_m),$$
where $r$ is the tree that  results from removing $k_m$ and $v_m$ from $t$.  The function is defined via the following equations:
  - $\texttt{Node}(k, v, \texttt{Nil}, r).\texttt{delMin}() = (r, k, v)$

    If the left subtree is empty, $k$ has to be the smallest key in the tree 
    $\texttt{Node}(k, v, \texttt{Nil}, r)$.  If $k$ is removed, we are left with the subtree $r$.
  - $l\not= \texttt{Nil} \wedge l.\texttt{delMin}() = (l',k_{min}, v_{min}) \;\rightarrow
      \texttt{Node}(k, v, l, r).\texttt{delMin}() = \bigl(\texttt{Node}(k, v, l', r), k_{min}, v_{min}\bigr)$.

    If the left subtree $l$ in the binary tree $t = \texttt{Node}(k, v, l, r)$
    is not empty, then the smallest key of  $t$ is located inside the left subtree $l$.
    This smallest key is recursively removed from  $l$. This yields the tree 
    $l'$.  Next,  $l$ is replaced by $l'$ in $t$.  The resulting tree is
    $t' = \texttt{Node}(k, v, l', r)$.

In [8]:
OrderedBinaryTree.prototype._delMin = function<Key, Value>(this: OrderedBinaryTree<Key, Value>): [OrderedBinaryTree<Key, Value>, Key, Value] {
    if (this.mLeft.isEmpty()) {
        return [this.mRight, this.mKey, this.mValue];
    } else {
        const [ls, km, vm] = this.mLeft!._delMin();
        this.mLeft = ls;
        return [this, km, vm];
    }
};

[Function (anonymous)]


Given two ordered binary trees `s` and `t`, the expression `s._update(t)` overwrites the attributes of `s` with the corresponding attributes of `t`.

In [9]:
OrderedBinaryTree.prototype._update = function<Key, Value>(this: OrderedBinaryTree<Key, Value>, t: OrderedBinaryTree<Key, Value>): void {
    this.mKey = t.mKey;
    this.mValue = t.mValue;
    this.mLeft = t.mLeft;
    this.mRight = t.mRight;
};

[Function (anonymous)]


Given an ordered binary tree $b$, the method $b.\texttt{keyList}()$ returns the list of all keys occurring in $b$.
Note that this list has to be sorted ascendingly.

In [10]:
OrderedBinaryTree.prototype._keyList = function<Key, Value>(this: OrderedBinaryTree<Key, Value>): Key[] {
    if (this.isEmpty()) {
        return [];
    }
    return [...this.mLeft._keyList(), this.mKey, ...this.mRight._keyList()];
};

[Function (anonymous)]


In [11]:
import { Graphviz } from "@hpcc-js/wasm";

Given a binary tree `t` the method `t._assignIDs(NodeDict)` assigns a unique identifier with each node.  The dictionary `NodeDict` maps these identifiers to the nodes where they occur.

In [12]:
OrderedBinaryTree.prototype._assignIDs = function<Key, Value>(this: OrderedBinaryTree<Key, Value>, nodeDict: Record<number, OrderedBinaryTree<Key, Value>>): void {
    OrderedBinaryTree.sNodeCount += 1;
    this.mID = OrderedBinaryTree.sNodeCount;
    nodeDict[this.mID] = this;
    if (this.isEmpty()) {
        return;
    }
    this.mLeft._assignIDs(nodeDict);
    this.mRight._assignIDs(nodeDict);
};

[Function (anonymous)]


Given an ordered binary tree $t$, the function $t.\texttt{toDot}()$ renders the tree graphically using `graphviz`.

In [13]:
OrderedBinaryTree.prototype.toDot = function<Key, Value>(this: OrderedBinaryTree<Key, Value>): string {
    OrderedBinaryTree.sNodeCount = 0;
    let dot = `digraph G {\nnode [shape=record style=rounded];\n`;
    const nodeDict: Record<number, OrderedBinaryTree<Key, Value>> = {};
    this._assignIDs(nodeDict);
    
    for (const [n, t] of Object.entries(nodeDict)) {
        if (t.mValue !== null) {
            dot += `${n} [label="{${t.mKey}|${t.mValue}}"];\n`;
        } else if (t.mKey !== null) {
            dot += `${n} [label="${t.mKey}"];\n`;
        } else {
            dot += `${n} [label="" shape=point];\n`;
        }
    }
    
    for (const [n, t] of Object.entries(nodeDict)) {
        if (t.mLeft !== null) {
            dot += `${n} -> ${t.mLeft.mID};\n`;
        }
        if (t.mRight !== null) {
            dot += `${n} -> ${t.mRight.mID};\n`;
        }
    }
    
    dot += `}\n`;
    return dot;
};

[Function (anonymous)]


The function $\texttt{demo}()$ creates a small ordered binary tree.

In [14]:
function demo(): OrderedBinaryTree<string, number> {
    const m = new OrderedBinaryTree<string, number>();
    m.insert("anton", 123);
    m.insert("hugo", 345);
    m.insert("gustav", 789);
    m.insert("mariam", 345);
    m.insert("jens", 234);
    m.insert("hubert", 432);
    m.insert("andre", 342);
    m.insert("philipp", 342);
    m.insert("rene", 345);
    m.insert("ans", 123);
    m.insert("alfa", 123);
    m.insert("algo", 345);
    return m;
}

In [15]:
import { display } from "tslab";

async function renderTree<Key, Value>(tree: OrderedBinaryTree<Key, Value>): Promise<void> {
    const dot = await tree.toDot();
    const gv = await Graphviz.load();
    const svg = gv.layout(dot, "svg", "dot");
    display.html(svg);
}

In [16]:
const t = demo();
await renderTree(t);

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.0.1 (0)
 -->
<!-- Title: G Pages: 1 -->
 
 
 G 
 
<!-- 1 -->
 
 1 
 
 anton 
 
 123 
 
<!-- 2 -->
 
 2 
 
 andre 
 
 342 
 
<!-- 1->2 -->
 
 1->2 
 
 
 
<!-- 11 -->
 
 11 
 
 hugo 
 
 345 
 
<!-- 1->11 -->
 
 1->11 
 
 
 
<!-- 3 -->
 
 3 
 
 alfa 
 
 123 
 
<!-- 2->3 -->
 
 2->3 
 
 
 
<!-- 8 -->
 
 8 
 
 ans 
 
 123 
 
<!-- 2->8 -->
 
 2->8 
 
 
 
<!-- 4 -->
 
 4 
 
 
<!-- 3->4 -->
 
 3->4 
 
 
 
<!-- 5 -->
 
 5 
 
 algo 
 
 345 
 
<!-- 3->5 -->
 
 3->5 
 
 
 
<!-- 6 -->
 
 6 
 
 
<!-- 5->6 -->
 
 5->6 
 
 
 
<!-- 7 -->
 
 7 
 
 
<!-- 5->7 -->
 
 5->7 
 
 
 
<!-- 9 -->
 
 9 
 
 
<!-- 8->9 -->
 
 8->9 
 
 
 
<!-- 10 -->
 
 10 
 
 
<!-- 8->10 -->
 
 8->10 
 
 
 
<!-- 12 -->
 
 12 
 
 gustav 
 
 789 
 
<!-- 11->12 -->
 
 11->12 
 
 
 
<!-- 17 -->
 
 17 
 
 mariam 
 
 345 
 
<!-- 11->17 -->
 
 11->17 
 
 
 
<!-- 13 -->
 
 13 
 
 
<!-- 12->13 -->
 
 12->13 
 
 
 
<!-- 14 -->
 
 14 
 
 hubert 
 
 432 
 
<!-- 12->14 -->
 
 12->14 
 
 
 
<!-- 15 -->
 
 15 
 
 
<!-- 14->15 -->
 
 14->15 
 
 
 
<!-- 16 -->
 
 16 
 
 
<!-- 14->16 -->
 
 14->16 
 
 
 
<!-- 18 -->
 
 18 
 
 jens 
 
 234 
 
<!-- 17->18 -->
 
 17->18 
 
 
 
<!-- 21 -->
 
 21 
 
 philipp 
 
 342 
 
<!-- 17->21 -->
 
 17->21 
 
 
 
<!-- 19 -->
 
 19 
 
 
<!-- 18->19 -->
 
 18->19 
 
 
 
<!-- 20 -->
 
 20 
 
 
<!-- 18->20 -->
 
 18->20 
 
 
 
<!-- 22 -->
 
 22 
 
 
<!-- 21->22 -->
 
 21->22 
 
 
 
<!-- 23 -->
 
 23 
 
 rene 
 
 345 
 
<!-- 21->23 -->
 
 21->23 
 
 
 
<!-- 24 -->
 
 24 
 
 
<!-- 23->24 -->
 
 23->24 
 
 
 
<!-- 25 -->
 
 25 
 
 
<!-- 23->25 -->
 
 23->25

In [17]:
t.insert('maurice', 123);
await renderTree(t);

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.0.1 (0)
 -->
<!-- Title: G Pages: 1 -->
 
 
 G 
 
<!-- 1 -->
 
 1 
 
 anton 
 
 123 
 
<!-- 2 -->
 
 2 
 
 andre 
 
 342 
 
<!-- 1->2 -->
 
 1->2 
 
 
 
<!-- 11 -->
 
 11 
 
 hugo 
 
 345 
 
<!-- 1->11 -->
 
 1->11 
 
 
 
<!-- 3 -->
 
 3 
 
 alfa 
 
 123 
 
<!-- 2->3 -->
 
 2->3 
 
 
 
<!-- 8 -->
 
 8 
 
 ans 
 
 123 
 
<!-- 2->8 -->
 
 2->8 
 
 
 
<!-- 4 -->
 
 4 
 
 
<!-- 3->4 -->
 
 3->4 
 
 
 
<!-- 5 -->
 
 5 
 
 algo 
 
 345 
 
<!-- 3->5 -->
 
 3->5 
 
 
 
<!-- 6 -->
 
 6 
 
 
<!-- 5->6 -->
 
 5->6 
 
 
 
<!-- 7 -->
 
 7 
 
 
<!-- 5->7 -->
 
 5->7 
 
 
 
<!-- 9 -->
 
 9 
 
 
<!-- 8->9 -->
 
 8->9 
 
 
 
<!-- 10 -->
 
 10 
 
 
<!-- 8->10 -->
 
 8->10 
 
 
 
<!-- 12 -->
 
 12 
 
 gustav 
 
 789 
 
<!-- 11->12 -->
 
 11->12 
 
 
 
<!-- 17 -->
 
 17 
 
 mariam 
 
 345 
 
<!-- 11->17 -->
 
 11->17 
 
 
 
<!-- 13 -->
 
 13 
 
 
<!-- 12->13 -->
 
 12->13 
 
 
 
<!-- 14 -->
 
 14 
 
 hubert 
 
 432 
 
<!-- 12->14 -->
 
 12->14 
 
 
 
<!-- 15 -->
 
 15 
 
 
<!-- 14->15 -->
 
 14->15 
 
 
 
<!-- 16 -->
 
 16 
 
 
<!-- 14->16 -->
 
 14->16 
 
 
 
<!-- 18 -->
 
 18 
 
 jens 
 
 234 
 
<!-- 17->18 -->
 
 17->18 
 
 
 
<!-- 21 -->
 
 21 
 
 philipp 
 
 342 
 
<!-- 17->21 -->
 
 17->21 
 
 
 
<!-- 19 -->
 
 19 
 
 
<!-- 18->19 -->
 
 18->19 
 
 
 
<!-- 20 -->
 
 20 
 
 
<!-- 18->20 -->
 
 18->20 
 
 
 
<!-- 22 -->
 
 22 
 
 maurice 
 
 123 
 
<!-- 21->22 -->
 
 21->22 
 
 
 
<!-- 25 -->
 
 25 
 
 rene 
 
 345 
 
<!-- 21->25 -->
 
 21->25 
 
 
 
<!-- 23 -->
 
 23 
 
 
<!-- 22->23 -->
 
 22->23 
 
 
 
<!-- 24 -->
 
 24 
 
 
<!-- 22->24 -->
 
 22->24 
 
 
 
<!-- 26 -->
 
 26 
 
 
<!-- 25->26 -->
 
 25->26 
 
 
 
<!-- 27 -->
 
 27 
 
 
<!-- 25->27 -->
 
 25->27

In [18]:
t.delete('mariam');
await renderTree(t);

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.0.1 (0)
 -->
<!-- Title: G Pages: 1 -->
 
 
 G 
 
<!-- 1 -->
 
 1 
 
 anton 
 
 123 
 
<!-- 2 -->
 
 2 
 
 andre 
 
 342 
 
<!-- 1->2 -->
 
 1->2 
 
 
 
<!-- 11 -->
 
 11 
 
 hugo 
 
 345 
 
<!-- 1->11 -->
 
 1->11 
 
 
 
<!-- 3 -->
 
 3 
 
 alfa 
 
 123 
 
<!-- 2->3 -->
 
 2->3 
 
 
 
<!-- 8 -->
 
 8 
 
 ans 
 
 123 
 
<!-- 2->8 -->
 
 2->8 
 
 
 
<!-- 4 -->
 
 4 
 
 
<!-- 3->4 -->
 
 3->4 
 
 
 
<!-- 5 -->
 
 5 
 
 algo 
 
 345 
 
<!-- 3->5 -->
 
 3->5 
 
 
 
<!-- 6 -->
 
 6 
 
 
<!-- 5->6 -->
 
 5->6 
 
 
 
<!-- 7 -->
 
 7 
 
 
<!-- 5->7 -->
 
 5->7 
 
 
 
<!-- 9 -->
 
 9 
 
 
<!-- 8->9 -->
 
 8->9 
 
 
 
<!-- 10 -->
 
 10 
 
 
<!-- 8->10 -->
 
 8->10 
 
 
 
<!-- 12 -->
 
 12 
 
 gustav 
 
 789 
 
<!-- 11->12 -->
 
 11->12 
 
 
 
<!-- 17 -->
 
 17 
 
 maurice 
 
 123 
 
<!-- 11->17 -->
 
 11->17 
 
 
 
<!-- 13 -->
 
 13 
 
 
<!-- 12->13 -->
 
 12->13 
 
 
 
<!-- 14 -->
 
 14 
 
 hubert 
 
 432 
 
<!-- 12->14 -->
 
 12->14 
 
 
 
<!-- 15 -->
 
 15 
 
 
<!-- 14->15 -->
 
 14->15 
 
 
 
<!-- 16 -->
 
 16 
 
 
<!-- 14->16 -->
 
 14->16 
 
 
 
<!-- 18 -->
 
 18 
 
 jens 
 
 234 
 
<!-- 17->18 -->
 
 17->18 
 
 
 
<!-- 21 -->
 
 21 
 
 philipp 
 
 342 
 
<!-- 17->21 -->
 
 17->21 
 
 
 
<!-- 19 -->
 
 19 
 
 
<!-- 18->19 -->
 
 18->19 
 
 
 
<!-- 20 -->
 
 20 
 
 
<!-- 18->20 -->
 
 18->20 
 
 
 
<!-- 22 -->
 
 22 
 
 
<!-- 21->22 -->
 
 21->22 
 
 
 
<!-- 23 -->
 
 23 
 
 rene 
 
 345 
 
<!-- 21->23 -->
 
 21->23 
 
 
 
<!-- 24 -->
 
 24 
 
 
<!-- 23->24 -->
 
 23->24 
 
 
 
<!-- 25 -->
 
 25 
 
 
<!-- 23->25 -->
 
 23->25

In [19]:
t.delete('anton');
await renderTree(t);

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.0.1 (0)
 -->
<!-- Title: G Pages: 1 -->
 
 
 G 
 
<!-- 1 -->
 
 1 
 
 gustav 
 
 789 
 
<!-- 2 -->
 
 2 
 
 andre 
 
 342 
 
<!-- 1->2 -->
 
 1->2 
 
 
 
<!-- 11 -->
 
 11 
 
 hugo 
 
 345 
 
<!-- 1->11 -->
 
 1->11 
 
 
 
<!-- 3 -->
 
 3 
 
 alfa 
 
 123 
 
<!-- 2->3 -->
 
 2->3 
 
 
 
<!-- 8 -->
 
 8 
 
 ans 
 
 123 
 
<!-- 2->8 -->
 
 2->8 
 
 
 
<!-- 4 -->
 
 4 
 
 
<!-- 3->4 -->
 
 3->4 
 
 
 
<!-- 5 -->
 
 5 
 
 algo 
 
 345 
 
<!-- 3->5 -->
 
 3->5 
 
 
 
<!-- 6 -->
 
 6 
 
 
<!-- 5->6 -->
 
 5->6 
 
 
 
<!-- 7 -->
 
 7 
 
 
<!-- 5->7 -->
 
 5->7 
 
 
 
<!-- 9 -->
 
 9 
 
 
<!-- 8->9 -->
 
 8->9 
 
 
 
<!-- 10 -->
 
 10 
 
 
<!-- 8->10 -->
 
 8->10 
 
 
 
<!-- 12 -->
 
 12 
 
 hubert 
 
 432 
 
<!-- 11->12 -->
 
 11->12 
 
 
 
<!-- 15 -->
 
 15 
 
 maurice 
 
 123 
 
<!-- 11->15 -->
 
 11->15 
 
 
 
<!-- 13 -->
 
 13 
 
 
<!-- 12->13 -->
 
 12->13 
 
 
 
<!-- 14 -->
 
 14 
 
 
<!-- 12->14 -->
 
 12->14 
 
 
 
<!-- 16 -->
 
 16 
 
 jens 
 
 234 
 
<!-- 15->16 -->
 
 15->16 
 
 
 
<!-- 19 -->
 
 19 
 
 philipp 
 
 342 
 
<!-- 15->19 -->
 
 15->19 
 
 
 
<!-- 17 -->
 
 17 
 
 
<!-- 16->17 -->
 
 16->17 
 
 
 
<!-- 18 -->
 
 18 
 
 
<!-- 16->18 -->
 
 16->18 
 
 
 
<!-- 20 -->
 
 20 
 
 
<!-- 19->20 -->
 
 19->20 
 
 
 
<!-- 21 -->
 
 21 
 
 rene 
 
 345 
 
<!-- 19->21 -->
 
 19->21 
 
 
 
<!-- 22 -->
 
 22 
 
 
<!-- 21->22 -->
 
 21->22 
 
 
 
<!-- 23 -->
 
 23 
 
 
<!-- 21->23 -->
 
 21->23

In [20]:
t.delete('gustav');
await renderTree(t);

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.0.1 (0)
 -->
<!-- Title: G Pages: 1 -->
 
 
 G 
 
<!-- 1 -->
 
 1 
 
 hubert 
 
 432 
 
<!-- 2 -->
 
 2 
 
 andre 
 
 342 
 
<!-- 1->2 -->
 
 1->2 
 
 
 
<!-- 11 -->
 
 11 
 
 hugo 
 
 345 
 
<!-- 1->11 -->
 
 1->11 
 
 
 
<!-- 3 -->
 
 3 
 
 alfa 
 
 123 
 
<!-- 2->3 -->
 
 2->3 
 
 
 
<!-- 8 -->
 
 8 
 
 ans 
 
 123 
 
<!-- 2->8 -->
 
 2->8 
 
 
 
<!-- 4 -->
 
 4 
 
 
<!-- 3->4 -->
 
 3->4 
 
 
 
<!-- 5 -->
 
 5 
 
 algo 
 
 345 
 
<!-- 3->5 -->
 
 3->5 
 
 
 
<!-- 6 -->
 
 6 
 
 
<!-- 5->6 -->
 
 5->6 
 
 
 
<!-- 7 -->
 
 7 
 
 
<!-- 5->7 -->
 
 5->7 
 
 
 
<!-- 9 -->
 
 9 
 
 
<!-- 8->9 -->
 
 8->9 
 
 
 
<!-- 10 -->
 
 10 
 
 
<!-- 8->10 -->
 
 8->10 
 
 
 
<!-- 12 -->
 
 12 
 
 
<!-- 11->12 -->
 
 11->12 
 
 
 
<!-- 13 -->
 
 13 
 
 maurice 
 
 123 
 
<!-- 11->13 -->
 
 11->13 
 
 
 
<!-- 14 -->
 
 14 
 
 jens 
 
 234 
 
<!-- 13->14 -->
 
 13->14 
 
 
 
<!-- 17 -->
 
 17 
 
 philipp 
 
 342 
 
<!-- 13->17 -->
 
 13->17 
 
 
 
<!-- 15 -->
 
 15 
 
 
<!-- 14->15 -->
 
 14->15 
 
 
 
<!-- 16 -->
 
 16 
 
 
<!-- 14->16 -->
 
 14->16 
 
 
 
<!-- 18 -->
 
 18 
 
 
<!-- 17->18 -->
 
 17->18 
 
 
 
<!-- 19 -->
 
 19 
 
 rene 
 
 345 
 
<!-- 17->19 -->
 
 17->19 
 
 
 
<!-- 20 -->
 
 20 
 
 
<!-- 19->20 -->
 
 19->20 
 
 
 
<!-- 21 -->
 
 21 
 
 
<!-- 19->21 -->
 
 19->21

In [21]:
t.delete('hubert');
await renderTree(t);

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.0.1 (0)
 -->
<!-- Title: G Pages: 1 -->
 
 
 G 
 
<!-- 1 -->
 
 1 
 
 hugo 
 
 345 
 
<!-- 2 -->
 
 2 
 
 andre 
 
 342 
 
<!-- 1->2 -->
 
 1->2 
 
 
 
<!-- 11 -->
 
 11 
 
 maurice 
 
 123 
 
<!-- 1->11 -->
 
 1->11 
 
 
 
<!-- 3 -->
 
 3 
 
 alfa 
 
 123 
 
<!-- 2->3 -->
 
 2->3 
 
 
 
<!-- 8 -->
 
 8 
 
 ans 
 
 123 
 
<!-- 2->8 -->
 
 2->8 
 
 
 
<!-- 4 -->
 
 4 
 
 
<!-- 3->4 -->
 
 3->4 
 
 
 
<!-- 5 -->
 
 5 
 
 algo 
 
 345 
 
<!-- 3->5 -->
 
 3->5 
 
 
 
<!-- 6 -->
 
 6 
 
 
<!-- 5->6 -->
 
 5->6 
 
 
 
<!-- 7 -->
 
 7 
 
 
<!-- 5->7 -->
 
 5->7 
 
 
 
<!-- 9 -->
 
 9 
 
 
<!-- 8->9 -->
 
 8->9 
 
 
 
<!-- 10 -->
 
 10 
 
 
<!-- 8->10 -->
 
 8->10 
 
 
 
<!-- 12 -->
 
 12 
 
 jens 
 
 234 
 
<!-- 11->12 -->
 
 11->12 
 
 
 
<!-- 15 -->
 
 15 
 
 philipp 
 
 342 
 
<!-- 11->15 -->
 
 11->15 
 
 
 
<!-- 13 -->
 
 13 
 
 
<!-- 12->13 -->
 
 12->13 
 
 
 
<!-- 14 -->
 
 14 
 
 
<!-- 12->14 -->
 
 12->14 
 
 
 
<!-- 16 -->
 
 16 
 
 
<!-- 15->16 -->
 
 15->16 
 
 
 
<!-- 17 -->
 
 17 
 
 rene 
 
 345 
 
<!-- 15->17 -->
 
 15->17 
 
 
 
<!-- 18 -->
 
 18 
 
 
<!-- 17->18 -->
 
 17->18 
 
 
 
<!-- 19 -->
 
 19 
 
 
<!-- 17->19 -->
 
 17->19

Let's generate an ordered binary tree with random keys.

In [22]:
const t = new OrderedBinaryTree<number, null>();
const rnd = {
    range: (max: number) => Math.floor(Math.random() * max)
};
for (let i = 0; i < 30; i++) {
    const k = rnd.range(100);
    t.insert(k, null);
}
await renderTree(t);

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.0.1 (0)
 -->
<!-- Title: G Pages: 1 -->
 
 
 G 
 
<!-- 1 -->
 
 1 
 
 1 
 
<!-- 2 -->
 
 2 
 
 
<!-- 1->2 -->
 
 1->2 
 
 
 
<!-- 3 -->
 
 3 
 
 14 
 
<!-- 1->3 -->
 
 1->3 
 
 
 
<!-- 4 -->
 
 4 
 
 10 
 
<!-- 3->4 -->
 
 3->4 
 
 
 
<!-- 7 -->
 
 7 
 
 53 
 
<!-- 3->7 -->
 
 3->7 
 
 
 
<!-- 5 -->
 
 5 
 
 
<!-- 4->5 -->
 
 4->5 
 
 
 
<!-- 6 -->
 
 6 
 
 
<!-- 4->6 -->
 
 4->6 
 
 
 
<!-- 8 -->
 
 8 
 
 47 
 
<!-- 7->8 -->
 
 7->8 
 
 
 
<!-- 25 -->
 
 25 
 
 97 
 
<!-- 7->25 -->
 
 7->25 
 
 
 
<!-- 9 -->
 
 9 
 
 35 
 
<!-- 8->9 -->
 
 8->9 
 
 
 
<!-- 22 -->
 
 22 
 
 48 
 
<!-- 8->22 -->
 
 8->22 
 
 
 
<!-- 10 -->
 
 10 
 
 
<!-- 9->10 -->
 
 9->10 
 
 
 
<!-- 11 -->
 
 11 
 
 37 
 
<!-- 9->11 -->
 
 9->11 
 
 
 
<!-- 12 -->
 
 12 
 
 
<!-- 11->12 -->
 
 11->12 
 
 
 
<!-- 13 -->
 
 13 
 
 39 
 
<!-- 11->13 -->
 
 11->13 
 
 
 
<!-- 14 -->
 
 14 
 
 
<!-- 13->14 -->
 
 13->14 
 
 
 
<!-- 15 -->
 
 15 
 
 43 
 
<!-- 13->15 -->
 
 13->15 
 
 
 
<!-- 16 -->
 
 16 
 
 
<!-- 15->16 -->
 
 15->16 
 
 
 
<!-- 17 -->
 
 17 
 
 44 
 
<!-- 15->17 -->
 
 15->17 
 
 
 
<!-- 18 -->
 
 18 
 
 
<!-- 17->18 -->
 
 17->18 
 
 
 
<!-- 19 -->
 
 19 
 
 45 
 
<!-- 17->19 -->
 
 17->19 
 
 
 
<!-- 20 -->
 
 20 
 
 
<!-- 19->20 -->
 
 19->20 
 
 
 
<!-- 21 -->
 
 21 
 
 
<!-- 19->21 -->
 
 19->21 
 
 
 
<!-- 23 -->
 
 23 
 
 
<!-- 22->23 -->
 
 22->23 
 
 
 
<!-- 24 -->
 
 24 
 
 
<!-- 22->24 -->
 
 22->24 
 
 
 
<!-- 26 -->
 
 26 
 
 94 
 
<!-- 25->26 -->
 
 25->26 
 
 
 
<!-- 49 -->
 
 49 
 
 
<!-- 25->49 -->
 
 25->49 
 
 
 
<!-- 27 -->
 
 27 
 
 79 
 
<!-- 26->27 -->
 
 26->27 
 
 
 
<!-- 48 -->
 
 48 
 
 
<!-- 26->48 -->
 
 26->48 
 
 
 
<!-- 28 -->
 
 28 
 
 70 
 
<!-- 27->28 -->
 
 27->28 
 
 
 
<!-- 43 -->
 
 43 
 
 93 
 
<!-- 27->43 -->
 
 27->43 
 
 
 
<!-- 29 -->
 
 29 
 
 58 
 
<!-- 28->29 -->
 
 28->29 
 
 
 
<!-- 38 -->
 
 38 
 
 77 
 
<!-- 28->38 -->
 
 28->38 
 
 
 
<!-- 30 -->
 
 30 
 
 
<!-- 29->30 -->
 
 29->30 
 
 
 
<!-- 31 -->
 
 31 
 
 67 
 
<!-- 29->31 -->
 
 29->31 
 
 
 
<!-- 32 -->
 
 32 
 
 65 
 
<!-- 31->32 -->
 
 31->32 
 
 
 
<!-- 35 -->
 
 35 
 
 68 
 
<!-- 31->35 -->
 
 31->35 
 
 
 
<!-- 33 -->
 
 33 
 
 
<!-- 32->33 -->
 
 32->33 
 
 
 
<!-- 34 -->
 
 34 
 
 
<!-- 32->34 -->
 
 32->34 
 
 
 
<!-- 36 -->
 
 36 
 
 
<!-- 35->36 -->
 
 35->36 
 
 
 
<!-- 37 -->
 
 37 
 
 
<!-- 35->37 -->
 
 35->37 
 
 
 
<!-- 39 -->
 
 39 
 
 71 
 
<!-- 38->39 -->
 
 38->39 
 
 
 
<!-- 42 -->
 
 42 
 
 
<!-- 38->42 -->
 
 38->42 
 
 
 
<!-- 40 -->
 
 40 
 
 
<!-- 39->40 -->
 
 39->40 
 
 
 
<!-- 41 -->
 
 41 
 
 
<!-- 39->41 -->
 
 39->41 
 
 
 
<!-- 44 -->
 
 44 
 
 91 
 
<!-- 43->44 -->
 
 43->44 
 
 
 
<!-- 47 -->
 
 47 
 
 
<!-- 43->47 -->
 
 43->47 
 
 
 
<!-- 45 -->
 
 45 
 
 
<!-- 44->45 -->
 
 44->45 
 
 
 
<!-- 46 -->
 
 46 
 
 
<!-- 44->46 -->
 
 44->46

This tree looks more or less balanced.  Lets us create a tree where things do not work out that well.

In [23]:
const t = new OrderedBinaryTree<number, null>();
for (let k = 0; k < 30; k++) {
    t.insert(k, null);
}
await renderTree(t);

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.0.1 (0)
 -->
<!-- Title: G Pages: 1 -->
 
 
 G 
 
<!-- 1 -->
 
 1 
 
 0 
 
<!-- 2 -->
 
 2 
 
 
<!-- 1->2 -->
 
 1->2 
 
 
 
<!-- 3 -->
 
 3 
 
 1 
 
<!-- 1->3 -->
 
 1->3 
 
 
 
<!-- 4 -->
 
 4 
 
 
<!-- 3->4 -->
 
 3->4 
 
 
 
<!-- 5 -->
 
 5 
 
 2 
 
<!-- 3->5 -->
 
 3->5 
 
 
 
<!-- 6 -->
 
 6 
 
 
<!-- 5->6 -->
 
 5->6 
 
 
 
<!-- 7 -->
 
 7 
 
 3 
 
<!-- 5->7 -->
 
 5->7 
 
 
 
<!-- 8 -->
 
 8 
 
 
<!-- 7->8 -->
 
 7->8 
 
 
 
<!-- 9 -->
 
 9 
 
 4 
 
<!-- 7->9 -->
 
 7->9 
 
 
 
<!-- 10 -->
 
 10 
 
 
<!-- 9->10 -->
 
 9->10 
 
 
 
<!-- 11 -->
 
 11 
 
 5 
 
<!-- 9->11 -->
 
 9->11 
 
 
 
<!-- 12 -->
 
 12 
 
 
<!-- 11->12 -->
 
 11->12 
 
 
 
<!-- 13 -->
 
 13 
 
 6 
 
<!-- 11->13 -->
 
 11->13 
 
 
 
<!-- 14 -->
 
 14 
 
 
<!-- 13->14 -->
 
 13->14 
 
 
 
<!-- 15 -->
 
 15 
 
 7 
 
<!-- 13->15 -->
 
 13->15 
 
 
 
<!-- 16 -->
 
 16 
 
 
<!-- 15->16 -->
 
 15->16 
 
 
 
<!-- 17 -->
 
 17 
 
 8 
 
<!-- 15->17 -->
 
 15->17 
 
 
 
<!-- 18 -->
 
 18 
 
 
<!-- 17->18 -->
 
 17->18 
 
 
 
<!-- 19 -->
 
 19 
 
 9 
 
<!-- 17->19 -->
 
 17->19 
 
 
 
<!-- 20 -->
 
 20 
 
 
<!-- 19->20 -->
 
 19->20 
 
 
 
<!-- 21 -->
 
 21 
 
 10 
 
<!-- 19->21 -->
 
 19->21 
 
 
 
<!-- 22 -->
 
 22 
 
 
<!-- 21->22 -->
 
 21->22 
 
 
 
<!-- 23 -->
 
 23 
 
 11 
 
<!-- 21->23 -->
 
 21->23 
 
 
 
<!-- 24 -->
 
 24 
 
 
<!-- 23->24 -->
 
 23->24 
 
 
 
<!-- 25 -->
 
 25 
 
 12 
 
<!-- 23->25 -->
 
 23->25 
 
 
 
<!-- 26 -->
 
 26 
 
 
<!-- 25->26 -->
 
 25->26 
 
 
 
<!-- 27 -->
 
 27 
 
 13 
 
<!-- 25->27 -->
 
 25->27 
 
 
 
<!-- 28 -->
 
 28 
 
 
<!-- 27->28 -->
 
 27->28 
 
 
 
<!-- 29 -->
 
 29 
 
 14 
 
<!-- 27->29 -->
 
 27->29 
 
 
 
<!-- 30 -->
 
 30 
 
 
<!-- 29->30 -->
 
 29->30 
 
 
 
<!-- 31 -->
 
 31 
 
 15 
 
<!-- 29->31 -->
 
 29->31 
 
 
 
<!-- 32 -->
 
 32 
 
 
<!-- 31->32 -->
 
 31->32 
 
 
 
<!-- 33 -->
 
 33 
 
 16 
 
<!-- 31->33 -->
 
 31->33 
 
 
 
<!-- 34 -->
 
 34 
 
 
<!-- 33->34 -->
 
 33->34 
 
 
 
<!-- 35 -->
 
 35 
 
 17 
 
<!-- 33->35 -->
 
 33->35 
 
 
 
<!-- 36 -->
 
 36 
 
 
<!-- 35->36 -->
 
 35->36 
 
 
 
<!-- 37 -->
 
 37 
 
 18 
 
<!-- 35->37 -->
 
 35->37 
 
 
 
<!-- 38 -->
 
 38 
 
 
<!-- 37->38 -->
 
 37->38 
 
 
 
<!-- 39 -->
 
 39 
 
 19 
 
<!-- 37->39 -->
 
 37->39 
 
 
 
<!-- 40 -->
 
 40 
 
 
<!-- 39->40 -->
 
 39->40 
 
 
 
<!-- 41 -->
 
 41 
 
 20 
 
<!-- 39->41 -->
 
 39->41 
 
 
 
<!-- 42 -->
 
 42 
 
 
<!-- 41->42 -->
 
 41->42 
 
 
 
<!-- 43 -->
 
 43 
 
 21 
 
<!-- 41->43 -->
 
 41->43 
 
 
 
<!-- 44 -->
 
 44 
 
 
<!-- 43->44 -->
 
 43->44 
 
 
 
<!-- 45 -->
 
 45 
 
 22 
 
<!-- 43->45 -->
 
 43->45 
 
 
 
<!-- 46 -->
 
 46 
 
 
<!-- 45->46 -->
 
 45->46 
 
 
 
<!-- 47 -->
 
 47 
 
 23 
 
<!-- 45->47 -->
 
 45->47 
 
 
 
<!-- 48 -->
 
 48 
 
 
<!-- 47->48 -->
 
 47->48 
 
 
 
<!-- 49 -->
 
 49 
 
 24 
 
<!-- 47->49 -->
 
 47->49 
 
 
 
<!-- 50 -->
 
 50 
 
 
<!-- 49->50 -->
 
 49->50 
 
 
 
<!-- 51 -->
 
 51 
 
 25 
 
<!-- 49->51 -->
 
 49->51 
 
 
 
<!-- 52 -->
 
 52 
 
 
<!-- 51->52 -->
 
 51->52 
 
 
 
<!-- 53 -->
 
 53 
 
 26 
 
<!-- 51->53 -->
 
 51->53 
 
 
 
<!-- 54 -->
 
 54 
 
 
<!-- 53->54 -->
 
 53->54 
 
 
 
<!-- 55 -->
 
 55 
 
 27 
 
<!-- 53->55 -->
 
 53->55 
 
 
 
<!-- 56 -->
 
 56 
 
 
<!-- 55->56 -->
 
 55->56 
 
 
 
<!-- 57 -->
 
 57 
 
 28 
 
<!-- 55->57 -->
 
 55->57 
 
 
 
<!-- 58 -->
 
 58 
 
 
<!-- 57->58 -->
 
 57->58 
 
 
 
<!-- 59 -->
 
 59 
 
 29 
 
<!-- 57->59 -->
 
 57->59 
 
 
 
<!-- 60 -->
 
 60 
 
 
<!-- 59->60 -->
 
 59->60 
 
 
 
<!-- 61 -->
 
 61 
 
 
<!-- 59->61 -->
 
 59->61

In order to check whether the method `delete` works as expected, we try the following:

In [24]:
for (let k = 0; k < 30; k++) {
    t.delete(k);
}
await renderTree(t);

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.0.1 (0)
 -->
<!-- Title: G Pages: 1 -->
 
 
 G 
 
<!-- 1 -->
 
 1

Let us compute the set $S$ of prime numbers up to some given number $n\in\mathbb{N}$.  Mathematically, this set can be defined as
$$ S = \{ 2, \cdots, n \} - \bigl\{ p \cdot q \;\big|\; p, q \in \{2, \cdots, n \} \bigr\}. $$

In [25]:
const n = 100;
const S = new OrderedBinaryTree<number, null>();
const L = Array.from({ length: n - 1 }, (_, i) => i + 2);
function shuffle(array: number[]): void {
    for (let i = array.length - 1; i > 0; i--) {
        const j = Math.floor(Math.random() * (i + 1));
        [array[i], array[j]] = [array[j], array[i]];
    }
}
shuffle(L);
for (const x of L) {
    S.insert(x, null);
}
await renderTree(S);

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.0.1 (0)
 -->
<!-- Title: G Pages: 1 -->
 
 
 G 
 
<!-- 1 -->
 
 1 
 
 78 
 
<!-- 2 -->
 
 2 
 
 55 
 
<!-- 1->2 -->
 
 1->2 
 
 
 
<!-- 155 -->
 
 155 
 
 86 
 
<!-- 1->155 -->
 
 1->155 
 
 
 
<!-- 3 -->
 
 3 
 
 28 
 
<!-- 2->3 -->
 
 2->3 
 
 
 
<!-- 110 -->
 
 110 
 
 74 
 
<!-- 2->110 -->
 
 2->110 
 
 
 
<!-- 4 -->
 
 4 
 
 7 
 
<!-- 3->4 -->
 
 3->4 
 
 
 
<!-- 57 -->
 
 57 
 
 49 
 
<!-- 3->57 -->
 
 3->57 
 
 
 
<!-- 5 -->
 
 5 
 
 2 
 
<!-- 4->5 -->
 
 4->5 
 
 
 
<!-- 16 -->
 
 16 
 
 26 
 
<!-- 4->16 -->
 
 4->16 
 
 
 
<!-- 6 -->
 
 6 
 
 
<!-- 5->6 -->
 
 5->6 
 
 
 
<!-- 7 -->
 
 7 
 
 4 
 
<!-- 5->7 -->
 
 5->7 
 
 
 
<!-- 8 -->
 
 8 
 
 3 
 
<!-- 7->8 -->
 
 7->8 
 
 
 
<!-- 11 -->
 
 11 
 
 5 
 
<!-- 7->11 -->
 
 7->11 
 
 
 
<!-- 9 -->
 
 9 
 
 
<!-- 8->9 -->
 
 8->9 
 
 
 
<!-- 10 -->
 
 10 
 
 
<!-- 8->10 -->
 
 8->10 
 
 
 
<!-- 12 -->
 
 12 
 
 
<!-- 11->12 -->
 
 11->12 
 
 
 
<!-- 13 -->
 
 13 
 
 6 
 
<!-- 11->13 -->
 
 11->13 
 
 
 
<!-- 14 -->
 
 14 
 
 
<!-- 13->14 -->
 
 13->14 
 
 
 
<!-- 15 -->
 
 15 
 
 
<!-- 13->15 -->
 
 13->15 
 
 
 
<!-- 17 -->
 
 17 
 
 11 
 
<!-- 16->17 -->
 
 16->17 
 
 
 
<!-- 54 -->
 
 54 
 
 27 
 
<!-- 16->54 -->
 
 16->54 
 
 
 
<!-- 18 -->
 
 18 
 
 9 
 
<!-- 17->18 -->
 
 17->18 
 
 
 
<!-- 25 -->
 
 25 
 
 15 
 
<!-- 17->25 -->
 
 17->25 
 
 
 
<!-- 19 -->
 
 19 
 
 8 
 
<!-- 18->19 -->
 
 18->19 
 
 
 
<!-- 22 -->
 
 22 
 
 10 
 
<!-- 18->22 -->
 
 18->22 
 
 
 
<!-- 20 -->
 
 20 
 
 
<!-- 19->20 -->
 
 19->20 
 
 
 
<!-- 21 -->
 
 21 
 
 
<!-- 19->21 -->
 
 19->21 
 
 
 
<!-- 23 -->
 
 23 
 
 
<!-- 22->23 -->
 
 22->23 
 
 
 
<!-- 24 -->
 
 24 
 
 
<!-- 22->24 -->
 
 22->24 
 
 
 
<!-- 26 -->
 
 26 
 
 14 
 
<!-- 25->26 -->
 
 25->26 
 
 
 
<!-- 33 -->
 
 33 
 
 25 
 
<!-- 25->33 -->
 
 25->33 
 
 
 
<!-- 27 -->
 
 27 
 
 12 
 
<!-- 26->27 -->
 
 26->27 
 
 
 
<!-- 32 -->
 
 32 
 
 
<!-- 26->32 -->
 
 26->32 
 
 
 
<!-- 28 -->
 
 28 
 
 
<!-- 27->28 -->
 
 27->28 
 
 
 
<!-- 29 -->
 
 29 
 
 13 
 
<!-- 27->29 -->
 
 27->29 
 
 
 
<!-- 30 -->
 
 30 
 
 
<!-- 29->30 -->
 
 29->30 
 
 
 
<!-- 31 -->
 
 31 
 
 
<!-- 29->31 -->
 
 29->31 
 
 
 
<!-- 34 -->
 
 34 
 
 24 
 
<!-- 33->34 -->
 
 33->34 
 
 
 
<!-- 53 -->
 
 53 
 
 
<!-- 33->53 -->
 
 33->53 
 
 
 
<!-- 35 -->
 
 35 
 
 22 
 
<!-- 34->35 -->
 
 34->35 
 
 
 
<!-- 52 -->
 
 52 
 
 
<!-- 34->52 -->
 
 34->52 
 
 
 
<!-- 36 -->
 
 36 
 
 21 
 
<!-- 35->36 -->
 
 35->36 
 
 
 
<!-- 49 -->
 
 49 
 
 23 
 
<!-- 35->49 -->
 
 35->49 
 
 
 
<!-- 37 -->
 
 37 
 
 18 
 
<!-- 36->37 -->
 
 36->37 
 
 
 
<!-- 48 -->
 
 48 
 
 
<!-- 36->48 -->
 
 36->48 
 
 
 
<!-- 38 -->
 
 38 
 
 16 
 
<!-- 37->38 -->
 
 37->38 
 
 
 
<!-- 43 -->
 
 43 
 
 20 
 
<!-- 37->43 -->
 
 37->43 
 
 
 
<!-- 39 -->
 
 39 
 
 
<!-- 38->39 -->
 
 38->39 
 
 
 
<!-- 40 -->
 
 40 
 
 17 
 
<!-- 38->40 -->
 
 38->40 
 
 
 
<!-- 41 -->
 
 41 
 
 
<!-- 40->41 -->
 
 40->41 
 
 
 
<!-- 42 -->
 
 42 
 
 
<!-- 40->42 -->
 
 40->42 
 
 
 
<!-- 44 -->
 
 44 
 
 19 
 
<!-- 43->44 -->
 
 43->44 
 
 
 
<!-- 47 -->
 
 47 
 
 
<!-- 43->47 -->
 
 43->47 
 
 
 
<!-- 45 -->
 
 45 
 
 
<!-- 44->45 -->
 
 44->45 
 
 
 
<!-- 46 -->
 
 46 
 
 
<!-- 44->46 -->
 
 44->46 
 
 
 
<!-- 50 -->
 
 50 
 
 
<!-- 49->50 -->
 
 49->50 
 
 
 
<!-- 51 -->
 
 51 
 
 
<!-- 49->51 -->
 
 49->51 
 
 
 
<!-- 55 -->
 
 55 
 
 
<!-- 54->55 -->
 
 54->55 
 
 
 
<!-- 56 -->
 
 56 
 
 
<!-- 54->56 -->
 
 54->56 
 
 
 
<!-- 58 -->
 
 58 
 
 34 
 
<!-- 57->58 -->
 
 57->58 
 
 
 
<!-- 99 -->
 
 99 
 
 51 
 
<!-- 57->99 -->
 
 57->99 
 
 
 
<!-- 59 -->
 
 59 
 
 31 
 
<!-- 58->59 -->
 
 58->59 
 
 
 
<!-- 70 -->
 
 70 
 
 42 
 
<!-- 58->70 -->
 
 58->70 
 
 
 
<!-- 60 -->
 
 60 
 
 30 
 
<!-- 59->60 -->
 
 59->60 
 
 
 
<!-- 65 -->
 
 65 
 
 33 
 
<!-- 59->65 -->
 
 59->65 
 
 
 
<!-- 61 -->
 
 61 
 
 

In [26]:
for (let p = 2; p <= Math.floor(n / 2); p++) {
    for (let q = p; q <= Math.floor(n / p); q++) {
        S.delete(p * q);
    }
}
await renderTree(S);

<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
<!-- Generated by graphviz version 14.0.1 (0)
 -->
<!-- Title: G Pages: 1 -->
 
 
 G 
 
<!-- 1 -->
 
 1 
 
 79 
 
<!-- 2 -->
 
 2 
 
 59 
 
<!-- 1->2 -->
 
 1->2 
 
 
 
<!-- 45 -->
 
 45 
 
 89 
 
<!-- 1->45 -->
 
 1->45 
 
 
 
<!-- 3 -->
 
 3 
 
 29 
 
<!-- 2->3 -->
 
 2->3 
 
 
 
<!-- 36 -->
 
 36 
 
 67 
 
<!-- 2->36 -->
 
 2->36 
 
 
 
<!-- 4 -->
 
 4 
 
 7 
 
<!-- 3->4 -->
 
 3->4 
 
 
 
<!-- 23 -->
 
 23 
 
 53 
 
<!-- 3->23 -->
 
 3->23 
 
 
 
<!-- 5 -->
 
 5 
 
 2 
 
<!-- 4->5 -->
 
 4->5 
 
 
 
<!-- 12 -->
 
 12 
 
 11 
 
<!-- 4->12 -->
 
 4->12 
 
 
 
<!-- 6 -->
 
 6 
 
 
<!-- 5->6 -->
 
 5->6 
 
 
 
<!-- 7 -->
 
 7 
 
 5 
 
<!-- 5->7 -->
 
 5->7 
 
 
 
<!-- 8 -->
 
 8 
 
 3 
 
<!-- 7->8 -->
 
 7->8 
 
 
 
<!-- 11 -->
 
 11 
 
 
<!-- 7->11 -->
 
 7->11 
 
 
 
<!-- 9 -->
 
 9 
 
 
<!-- 8->9 -->
 
 8->9 
 
 
 
<!-- 10 -->
 
 10 
 
 
<!-- 8->10 -->
 
 8->10 
 
 
 
<!-- 13 -->
 
 13 
 
 
<!-- 12->13 -->
 
 12->13 
 
 
 
<!-- 14 -->
 
 14 
 
 17 
 
<!-- 12->14 -->
 
 12->14 
 
 
 
<!-- 15 -->
 
 15 
 
 13 
 
<!-- 14->15 -->
 
 14->15 
 
 
 
<!-- 18 -->
 
 18 
 
 23 
 
<!-- 14->18 -->
 
 14->18 
 
 
 
<!-- 16 -->
 
 16 
 
 
<!-- 15->16 -->
 
 15->16 
 
 
 
<!-- 17 -->
 
 17 
 
 
<!-- 15->17 -->
 
 15->17 
 
 
 
<!-- 19 -->
 
 19 
 
 19 
 
<!-- 18->19 -->
 
 18->19 
 
 
 
<!-- 22 -->
 
 22 
 
 
<!-- 18->22 -->
 
 18->22 
 
 
 
<!-- 20 -->
 
 20 
 
 
<!-- 19->20 -->
 
 19->20 
 
 
 
<!-- 21 -->
 
 21 
 
 
<!-- 19->21 -->
 
 19->21 
 
 
 
<!-- 24 -->
 
 24 
 
 37 
 
<!-- 23->24 -->
 
 23->24 
 
 
 
<!-- 35 -->
 
 35 
 
 
<!-- 23->35 -->
 
 23->35 
 
 
 
<!-- 25 -->
 
 25 
 
 31 
 
<!-- 24->25 -->
 
 24->25 
 
 
 
<!-- 28 -->
 
 28 
 
 43 
 
<!-- 24->28 -->
 
 24->28 
 
 
 
<!-- 26 -->
 
 26 
 
 
<!-- 25->26 -->
 
 25->26 
 
 
 
<!-- 27 -->
 
 27 
 
 
<!-- 25->27 -->
 
 25->27 
 
 
 
<!-- 29 -->
 
 29 
 
 41 
 
<!-- 28->29 -->
 
 28->29 
 
 
 
<!-- 32 -->
 
 32 
 
 47 
 
<!-- 28->32 -->
 
 28->32 
 
 
 
<!-- 30 -->
 
 30 
 
 
<!-- 29->30 -->
 
 29->30 
 
 
 
<!-- 31 -->
 
 31 
 
 
<!-- 29->31 -->
 
 29->31 
 
 
 
<!-- 33 -->
 
 33 
 
 
<!-- 32->33 -->
 
 32->33 
 
 
 
<!-- 34 -->
 
 34 
 
 
<!-- 32->34 -->
 
 32->34 
 
 
 
<!-- 37 -->
 
 37 
 
 61 
 
<!-- 36->37 -->
 
 36->37 
 
 
 
<!-- 40 -->
 
 40 
 
 71 
 
<!-- 36->40 -->
 
 36->40 
 
 
 
<!-- 38 -->
 
 38 
 
 
<!-- 37->38 -->
 
 37->38 
 
 
 
<!-- 39 -->
 
 39 
 
 
<!-- 37->39 -->
 
 37->39 
 
 
 
<!-- 41 -->
 
 41 
 
 
<!-- 40->41 -->
 
 40->41 
 
 
 
<!-- 42 -->
 
 42 
 
 73 
 
<!-- 40->42 -->
 
 40->42 
 
 
 
<!-- 43 -->
 
 43 
 
 
<!-- 42->43 -->
 
 42->43 
 
 
 
<!-- 44 -->
 
 44 
 
 
<!-- 42->44 -->
 
 42->44 
 
 
 
<!-- 46 -->
 
 46 
 
 83 
 
<!-- 45->46 -->
 
 45->46 
 
 
 
<!-- 49 -->
 
 49 
 
 97 
 
<!-- 45->49 -->
 
 45->49 
 
 
 
<!-- 47 -->
 
 47 
 
 
<!-- 46->47 -->
 
 46->47 
 
 
 
<!-- 48 -->
 
 48 
 
 
<!-- 46->48 -->
 
 46->48 
 
 
 
<!-- 50 -->
 
 50 
 
 
<!-- 49->50 -->
 
 49->50 
 
 
 
<!-- 51 -->
 
 51 
 
 
<!-- 49->51 -->
 
 49->51

In [27]:
console.log(S._keyList());

[
   2,  3,  5,  7, 11, 13, 17, 19,
  23, 29, 31, 37, 41, 43, 47, 53,
  59, 61, 67, 71, 73, 79, 83, 89,
  97
]
